# Webhook Ingestion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/webhook/webhook_demo.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/webhook/webhook_demo.ipynb)

## Business Scenario

SaaS platforms send webhooks for orders, payments, or alerts. You need to validate and store those events as they arrive.

## Value Proposition

- Simple HTTP ingestion with schema validation
- Consistent event formats
- Quarantine bad payloads

---

## Goals

1. Receive webhook events
2. Validate payloads
3. Persist clean events


## 🚀 Step 1: Examine the Contract

Our contract `webhook_contract.yaml` defines a listener on port `8080` at the `/github-webhook` path.

In [ ]:
with open('webhook_contract.yaml', 'r') as f:
    print("📄 Webhook Contract Content:")
    print("-------------------------")
    print(f.read())

## ▶️ Step 2: Start the Webhook Listener

Since the processor runs in a blocking loop, we will start it in a background thread so we can send events to it from this same notebook.

In [ ]:
from lakelogic.core.streaming_processor import StreamingDataProcessor
import threading
import time

# Initialize processor
processor = StreamingDataProcessor(contract="webhook_contract.yaml", framework="bytewax")

# Start in a background thread
thread = threading.Thread(target=processor.start)
thread.daemon = True
thread.start()

print("🚀 Webhook receiver started in background!")
print("Listening at http://localhost:8080/github-webhook")
time.sleep(2) # Give it a moment to bind to the port

## 🧪 Step 3: Simulate Incoming Events

Now we'll use the `requests` library to send sample payloads to our running LakeLogic receiver.

In [ ]:
import requests
import json

url = "http://localhost:8080/github-webhook"

events = [
    {"action": "opened", "issue": {"title": "Feature Request: GraphQL support"}, "sender": {"login": "alice"}},
    {"action": "commented", "issue": {"title": "Fix SQL injection"}, "sender": {"login": "bob"}},
    {"action": "closed", "issue": {"title": "Update README"}, "sender": {"login": "charlie"}}
]

print("🧪 Sending test webhooks...")
for event in events:
    print(f"📤 Sending {event['action']} event for: {event['issue']['title']}")
    response = requests.post(url, json=event)
    print(f"📥 Server response: {response.status_code} - {response.json()}")
    time.sleep(1)

print("\n✅ Simulation complete!")

## 🛑 Step 4: Cleanup

Stop the background server.

In [ ]:
if 'processor' in locals():
    print("👋 Shutting down webhook server...")
    # Note: In a real notebook you might need to interrupt the kernel 
    # but here we'll try to shut down the server gracefully if implemented
    if hasattr(processor, 'stop'):
        processor.stop()

## 🎉 Summary

You just:
- ✅ Configured a push-based data source (Webhook)
- ✅ Launched a real-time HTTP receiver
- ✅ Successfully processed incoming event notifications

This pattern is ideal for event-driven architectures where your data pipeline needs to respond instantly to external triggers!